In [36]:
import pandas as pd
import requests
import json
import musicbrainzngs
from sklearn.preprocessing import StandardScaler


In [2]:
# Inladen Excel-file
df = pd.read_excel("goud_master_experiment.xlsx")
df.head()

,track_id,track_name,artist_name,owner,album_name,playlist_name
0,0QxCLD4xJlE8vv0Cvrb8TO,'74-'75,The Connells,Astrid,NaN,NaN
1,0IR503DQ7KTbH3MoOQGCJu,10 Stories Down,Michelle Featherstone,Astrid,Blue Bike,Dear Diary
2,0CZ8lquoTX2Dkg7Ak2inwA,1950,King Princess,Astrid,NaN,NaN
3,530iu5ZbzPX8KFLMs7qSK2,1983,Stefano Guzzetti,Astrid,NaN,NaN
4,7129iqBafaphfc3WPCGC0L,7 Years,Lukas Graham,Astrid,NaN,NaN


In [3]:
# Neem 1 testtrack
row = df.iloc[0]

artist = row["artist_name"]
track = row["track_name"]

artist, track


('The Connells', "'74-'75")

In [4]:
def search_mbid(artist, track):
    """
    Zoek MusicBrainz ID (MBID) op basis van artiest + tracknaam.
    """
    query = f'"{track}" AND artist:"{artist}"'
    url = f"https://musicbrainz.org/ws/2/recording/?query={query}&fmt=json"

    try:
        r = requests.get(url, headers={"User-Agent": "Astrid-Research-Tool"})
        data = r.json()

        # Als er resultaten zijn, neem de eerste
        if "recordings" in data and len(data["recordings"]) > 0:
            return data["recordings"][0]["id"]

        return None

    except Exception as e:
        print("Fout bij MBID lookup:", e)
        return None


In [5]:
mbid = search_mbid(artist, track)
mbid


Fout bij MBID lookup: ('Connection aborted.', ConnectionResetError(10054, 'An existing connection was forcibly closed by the remote host', None, 10054, None))


In [6]:
def get_acousticbrainz_features(mbid):
    """
    Haal low-level audio features op via AcousticBrainz mirror.
    """
    url = f"https://acousticbrainz.org/api/v1/{mbid}/low-level"

    try:
        r = requests.get(url)
        if r.status_code != 200:
            print("Geen AcousticBrainz data gevonden (status:", r.status_code, ")")
            return None
        return r.json()
    except Exception as e:
        print("Fout bij AcousticBrainz:", e)
        return None


In [7]:
features = get_acousticbrainz_features(mbid)
features


Geen AcousticBrainz data gevonden (status: 404 )


In [8]:
# Extract features uit AcousticBrainz JSON

def extract_features(ab_data):
    if ab_data is None:
        return None

    try:
        ll = ab_data["lowlevel"]

        return {
            "average_loudness": ll.get("average_loudness"),
            "tuning_frequency": ll.get("tuning_frequency"),
            "tuning_diatonic_strength": ll.get("tuning_diatonic_strength"),
            "tuning_equal_tempered_deviation": ll.get("tuning_equal_tempered_deviation"),
            "tuning_nontempered_energy_ratio": ll.get("tuning_nontempered_energy_ratio"),
            "spectral_centroid": ll.get("spectral", {}).get("centroid", {}).get("mean"),
            "mfcc_1": ll.get("mfcc", {}).get("mean", [None])[0],
            "mfcc_2": ll.get("mfcc", {}).get("mean", [None])[1],
        }
    except Exception as e:
        print("Extractie mislukt:", e)
        return None


In [9]:
extracted = extract_features(features)
extracted


### Beperkte testset - feature families en aggregated functions

In [10]:
# Volledige werk-kopie van de goud-data
work_df = df.copy()

# Eerste inspectie
work_df.head()



,track_id,track_name,artist_name,owner,album_name,playlist_name
0,0QxCLD4xJlE8vv0Cvrb8TO,'74-'75,The Connells,Astrid,NaN,NaN
1,0IR503DQ7KTbH3MoOQGCJu,10 Stories Down,Michelle Featherstone,Astrid,Blue Bike,Dear Diary
2,0CZ8lquoTX2Dkg7Ak2inwA,1950,King Princess,Astrid,NaN,NaN
3,530iu5ZbzPX8KFLMs7qSK2,1983,Stefano Guzzetti,Astrid,NaN,NaN
4,7129iqBafaphfc3WPCGC0L,7 Years,Lukas Graham,Astrid,NaN,NaN


In [11]:
# MusicBrainz client initialiseren
musicbrainzngs.set_useragent(
    "AstridAudioProject",
    "1.0",
    "https://example.com/contact"
)

In [12]:
def search_mbid(artist, track):
    try:
        query = f'artist:"{artist}" AND recording:"{track}"'
        result = musicbrainzngs.search_recordings(query=query, limit=1)

        if "recording-list" in result and len(result["recording-list"]) > 0:
            return result["recording-list"][0]["id"]
        else:
            return None

    except Exception as e:
        print(f"MBID lookup error for '{track}' by '{artist}': {e}")
        return None

search_mbid("The Connells", "'74-'75")


'bcb4d5d0-e8e2-4284-8a62-ec74f808a5de'

In [13]:
# MBID-kolom toevoegen aan work_df
work_df["mbid"] = None

for idx, row in work_df.iterrows():
    artist = row["artist_name"]
    track = row["track_name"]

    mbid = search_mbid(artist, track)  # onze verbeterde functie

    work_df.at[idx, "mbid"] = mbid

# Resultaat tonen
work_df[["track_name", "artist_name", "mbid"]]


MBID lookup error for '95' by 'Picture This': caused by: <urlopen error [WinError 10054] An existing connection was forcibly closed by the remote host>
MBID lookup error for 'A Mistake' by 'Fiona Apple': caused by: <urlopen error [WinError 10054] An existing connection was forcibly closed by the remote host>
MBID lookup error for 'All My Life' by 'Foo Fighters': caused by: <urlopen error [WinError 10054] An existing connection was forcibly closed by the remote host>
MBID lookup error for 'Ashes And Wine' by 'A Fine Frenzy': caused by: <urlopen error [WinError 10054] An existing connection was forcibly closed by the remote host>
MBID lookup error for 'Aurora' by 'Foo Fighters': caused by: <urlopen error [WinError 10054] An existing connection was forcibly closed by the remote host>
MBID lookup error for 'Austin (Boots Stop Workin')' by 'Dasha': caused by: <urlopen error [WinError 10054] An existing connection was forcibly closed by the remote host>
MBID lookup error for 'Best of You' by

,track_name,artist_name,mbid
0,'74-'75,The Connells,bcb4d5d0-e8e2-4284-8a62-ec74f808a5de
1,10 Stories Down,Michelle Featherstone,144e5fa4-c656-4c04-8792-3861adfd2fff
2,1950,King Princess,1685f3da-0ba4-454c-9680-0d33373822d0
3,1983,Stefano Guzzetti,43578605-c761-4d3e-b536-d59206163ff7
4,7 Years,Lukas Graham,cc31df03-b8fa-4e0f-96d2-5fbe4c19d7e7
...,...,...,...
781,lippy kids,Elbow,None
782,my tears ricochet,Taylor Swift,53b484ab-d9bd-412c-a00c-6351415af845
783,the lonely,Christina Perri,11695c51-cafd-4efe-a095-122b0838b41f
784,undressed,sombr,fb506141-c6c7-4b24-afe3-16c96bebd4bf


In [14]:
work_df["mbid"].notna().sum()


np.int64(465)

In [15]:
total_rows = len(work_df)
with_mbid = work_df["mbid"].count()
without_mbid = total_rows - with_mbid
coverage = with_mbid / total_rows * 100

print(f"Totaal aantal rijen: {total_rows}")
print(f"Met MBID: {with_mbid}")
print(f"Zonder MBID: {without_mbid}")
print(f"Dekking MBID: {coverage:.2f}%")


Totaal aantal rijen: 786
Met MBID: 465
Zonder MBID: 321
Dekking MBID: 59.16%


In [16]:
# Enkel tracks met MBID behouden
work_df_with_mbid = work_df[work_df["mbid"].notna()].copy()

# Inspectie
work_df_with_mbid.head()


,track_id,track_name,artist_name,owner,album_name,playlist_name,mbid
0,0QxCLD4xJlE8vv0Cvrb8TO,'74-'75,The Connells,Astrid,NaN,NaN,bcb4d5d0-e8e2-4284-8a62-ec74f808a5de
1,0IR503DQ7KTbH3MoOQGCJu,10 Stories Down,Michelle Featherstone,Astrid,Blue Bike,Dear Diary,144e5fa4-c656-4c04-8792-3861adfd2fff
2,0CZ8lquoTX2Dkg7Ak2inwA,1950,King Princess,Astrid,NaN,NaN,1685f3da-0ba4-454c-9680-0d33373822d0
3,530iu5ZbzPX8KFLMs7qSK2,1983,Stefano Guzzetti,Astrid,NaN,NaN,43578605-c761-4d3e-b536-d59206163ff7
4,7129iqBafaphfc3WPCGC0L,7 Years,Lukas Graham,Astrid,NaN,NaN,cc31df03-b8fa-4e0f-96d2-5fbe4c19d7e7


In [17]:
work_df_with_mbid.shape

(465, 7)

In [18]:
# Nieuwe kolom voor AcousticBrainz low-level data
work_df_with_mbid["ab_lowlevel"] = None

# Controle
work_df_with_mbid.head()


,track_id,track_name,artist_name,owner,album_name,playlist_name,mbid,ab_lowlevel
0,0QxCLD4xJlE8vv0Cvrb8TO,'74-'75,The Connells,Astrid,NaN,NaN,bcb4d5d0-e8e2-4284-8a62-ec74f808a5de,None
1,0IR503DQ7KTbH3MoOQGCJu,10 Stories Down,Michelle Featherstone,Astrid,Blue Bike,Dear Diary,144e5fa4-c656-4c04-8792-3861adfd2fff,None
2,0CZ8lquoTX2Dkg7Ak2inwA,1950,King Princess,Astrid,NaN,NaN,1685f3da-0ba4-454c-9680-0d33373822d0,None
3,530iu5ZbzPX8KFLMs7qSK2,1983,Stefano Guzzetti,Astrid,NaN,NaN,43578605-c761-4d3e-b536-d59206163ff7,None
4,7129iqBafaphfc3WPCGC0L,7 Years,Lukas Graham,Astrid,NaN,NaN,cc31df03-b8fa-4e0f-96d2-5fbe4c19d7e7,None


In [19]:
for idx, row in work_df_with_mbid.iterrows():
    mbid = row["mbid"]

    try:
        ab_data = get_acousticbrainz_features(mbid)
    except Exception as e:
        print(f"AB lookup error for MBID {mbid}: {e}")
        ab_data = None

    # Alleen de lowlevel-sectie bewaren
    if ab_data is not None and "lowlevel" in ab_data:
        work_df_with_mbid.at[idx, "ab_lowlevel"] = ab_data["lowlevel"]
    else:
        work_df_with_mbid.at[idx, "ab_lowlevel"] = None

# Inspectie
work_df_with_mbid[["track_name", "mbid", "ab_lowlevel"]].head()


Geen AcousticBrainz data gevonden (status: 404 )
Geen AcousticBrainz data gevonden (status: 404 )
Geen AcousticBrainz data gevonden (status: 404 )
Geen AcousticBrainz data gevonden (status: 404 )
Geen AcousticBrainz data gevonden (status: 404 )
Geen AcousticBrainz data gevonden (status: 404 )
Geen AcousticBrainz data gevonden (status: 404 )
Geen AcousticBrainz data gevonden (status: 404 )
Geen AcousticBrainz data gevonden (status: 404 )
Geen AcousticBrainz data gevonden (status: 404 )
Geen AcousticBrainz data gevonden (status: 404 )
Geen AcousticBrainz data gevonden (status: 404 )
Geen AcousticBrainz data gevonden (status: 404 )
Geen AcousticBrainz data gevonden (status: 404 )
Geen AcousticBrainz data gevonden (status: 404 )
Geen AcousticBrainz data gevonden (status: 404 )
Geen AcousticBrainz data gevonden (status: 404 )
Geen AcousticBrainz data gevonden (status: 404 )
Geen AcousticBrainz data gevonden (status: 404 )
Geen AcousticBrainz data gevonden (status: 404 )
Geen AcousticBrainz 

,track_name,mbid,ab_lowlevel
0,'74-'75,bcb4d5d0-e8e2-4284-8a62-ec74f808a5de,"{'average_loudness': 0.909114122391, 'barkband..."
1,10 Stories Down,144e5fa4-c656-4c04-8792-3861adfd2fff,None
2,1950,1685f3da-0ba4-454c-9680-0d33373822d0,None
3,1983,43578605-c761-4d3e-b536-d59206163ff7,None
4,7 Years,cc31df03-b8fa-4e0f-96d2-5fbe4c19d7e7,None


In [20]:
work_df_with_mbid["ab_lowlevel"].notna().sum()


np.int64(205)

In [21]:
# Enkel tracks met AB-data behouden
work_df_with_ab = work_df_with_mbid[work_df_with_mbid["ab_lowlevel"].notna()].copy()

# Inspectie
work_df_with_ab.head()


,track_id,track_name,artist_name,owner,album_name,playlist_name,mbid,ab_lowlevel
0,0QxCLD4xJlE8vv0Cvrb8TO,'74-'75,The Connells,Astrid,NaN,NaN,bcb4d5d0-e8e2-4284-8a62-ec74f808a5de,"{'average_loudness': 0.909114122391, 'barkband..."
8,2my0HGJw82GQrF3xhnTUVS,A Song for the Drunk and Broken Hearted,Passenger,Astrid,NaN,NaN,434dbfe7-a437-4030-87aa-36f020099909,"{'average_loudness': 0.810895383358, 'barkband..."
14,6sSDEFMeyf16WRDLRCXwTc,Absolute,The Fray,Astrid,The Fray,Dear Diary,04b1bd97-5039-439f-ab4d-5e50d8b0a51d,"{'average_loudness': 0.906621813774, 'barkband..."
17,3XzLC1TU451F0nPiJitk49,After Rain,Dermot Kennedy,Astrid,NaN,NaN,ac0abc3e-5aff-4baf-ad60-a1a66fad27cb,"{'average_loudness': 0.457140833139, 'barkband..."
21,3kb72STxc2959ZqsTwu52i,Ain't No Rest for the Wicked,Cage The Elephant,Astrid,NaN,NaN,e9bb1073-94e7-4ad0-a623-0bf0034af48c,"{'average_loudness': 0.80604916811, 'barkbands..."


In [22]:
work_df_with_ab.shape

(205, 8)

In [23]:
total = len(work_df)
with_mbid = len(work_df_with_mbid)
with_ab = len(work_df_with_ab)

# Percentages
pct_mbid = with_mbid / total * 100
pct_ab = with_ab / total * 100

# Tabel maken
loss_df = pd.DataFrame({
    "Stap": [
        "Originele dataset",
        "Met MBID",
        "Met AcousticBrainz data"
    ],
    "Aantal rijen": [
        total,
        with_mbid,
        with_ab
    ],
    "% van origineel": [
        100.0,
        pct_mbid,
        pct_ab
    ]
})

loss_df


,Stap,Aantal rijen,% van origineel
0,Originele dataset,786,100.000000
1,Met MBID,465,59.160305
2,Met AcousticBrainz data,205,26.081425


In [24]:
# Definitie van de features die we willen extraheren uit AB lowlevel
selected_features = {
    "spectral": [
        "spectral_centroid.mean",
        "spectral_centroid.var",
        "spectral_kurtosis.mean",
        "spectral_kurtosis.var",
        "spectral_skewness.mean",
        "spectral_skewness.var",
        "spectral_spread.mean",
        "spectral_spread.var"
    ],
    "mfcc_mean": [f"mfcc.mean[{i}]" for i in range(13)],
    "mfcc_var": [f"mfcc.var[{i}]" for i in range(13)],
    "loudness": [
        "loudness_ebu128.integrated",
        "loudness_ebu128.loudness_range",
        "dynamic_complexity"
    ],
    "rhythm": [
        "bpm",
        "beats_loudness.mean",
        "beats_loudness.var"
    ],
    "tonal": [
        "key_key",
        "key_scale",
        "key_strength"
    ]
}

selected_features


{'spectral': ['spectral_centroid.mean',
  'spectral_centroid.var',
  'spectral_kurtosis.mean',
  'spectral_kurtosis.var',
  'spectral_skewness.mean',
  'spectral_skewness.var',
  'spectral_spread.mean',
  'spectral_spread.var'],
 'mfcc_mean': ['mfcc.mean[0]',
  'mfcc.mean[1]',
  'mfcc.mean[2]',
  'mfcc.mean[3]',
  'mfcc.mean[4]',
  'mfcc.mean[5]',
  'mfcc.mean[6]',
  'mfcc.mean[7]',
  'mfcc.mean[8]',
  'mfcc.mean[9]',
  'mfcc.mean[10]',
  'mfcc.mean[11]',
  'mfcc.mean[12]'],
 'mfcc_var': ['mfcc.var[0]',
  'mfcc.var[1]',
  'mfcc.var[2]',
  'mfcc.var[3]',
  'mfcc.var[4]',
  'mfcc.var[5]',
  'mfcc.var[6]',
  'mfcc.var[7]',
  'mfcc.var[8]',
  'mfcc.var[9]',
  'mfcc.var[10]',
  'mfcc.var[11]',
  'mfcc.var[12]'],
 'loudness': ['loudness_ebu128.integrated',
  'loudness_ebu128.loudness_range',
  'dynamic_complexity'],
 'rhythm': ['bpm', 'beats_loudness.mean', 'beats_loudness.var'],
 'tonal': ['key_key', 'key_scale', 'key_strength']}

In [25]:
def extract_feature(lowlevel, path):
    """
    Haalt een feature op uit de nested AB lowlevel JSON.
    path is een string zoals 'mfcc.mean[3]' of 'spectral_centroid.mean'.
    Geeft None terug als het pad niet bestaat.
    """
    try:
        parts = path.split(".")
        obj = lowlevel

        for p in parts:
            # MFCC index zoals mfcc.mean[3]
            if "[" in p and "]" in p:
                base, idx = p.split("[")
                idx = int(idx[:-1])  # remove ]
                obj = obj[base][idx]
            else:
                obj = obj[p]

        return obj

    except Exception:
        return None


In [28]:
# Alle feature-namen flattenen naar één lijst
all_features = (
    selected_features["spectral"]
    + selected_features["mfcc_mean"]
    + selected_features["mfcc_var"]
    + selected_features["loudness"]
    + selected_features["rhythm"]
    + selected_features["tonal"]
)

# Dataframe voorbereiden
feature_df = pd.DataFrame(index=work_df_with_ab.index, columns=all_features)

feature_df.head()


,spectral_centroid.mean,spectral_centroid.var,spectral_kurtosis.mean,spectral_kurtosis.var,spectral_skewness.mean,spectral_skewness.var,spectral_spread.mean,spectral_spread.var,mfcc.mean[0],mfcc.mean[1],...,mfcc.var[12],loudness_ebu128.integrated,loudness_ebu128.loudness_range,dynamic_complexity,bpm,beats_loudness.mean,beats_loudness.var,key_key,key_scale,key_strength
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
14,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
17,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
21,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [30]:
for idx, row in work_df_with_ab.iterrows():
    low = row["ab_lowlevel"]

    for feat in all_features:
        feature_df.at[idx, feat] = extract_feature(low, feat)


In [31]:
feature_df.head()

,spectral_centroid.mean,spectral_centroid.var,spectral_kurtosis.mean,spectral_kurtosis.var,spectral_skewness.mean,spectral_skewness.var,spectral_spread.mean,spectral_spread.var,mfcc.mean[0],mfcc.mean[1],...,mfcc.var[12],loudness_ebu128.integrated,loudness_ebu128.loudness_range,dynamic_complexity,bpm,beats_loudness.mean,beats_loudness.var,key_key,key_scale,key_strength
0,927.821899,300542.6875,4.788329,19.162428,1.729906,0.271612,5182013.5,3974965231620.0,-649.453369,130.93927,...,None,None,None,3.81177,None,None,None,None,None,None
8,948.941101,472410.34375,6.854862,42.60194,1.874984,0.48039,4154438.75,4048178642940.0,-673.102356,146.645157,...,None,None,None,4.169648,None,None,None,None,None,None
14,1381.589844,322264.625,7.707166,421.949371,1.76829,1.684817,4820810.5,3929293717500.0,-616.204529,116.093132,...,None,None,None,2.615295,None,None,None,None,None,None
17,741.026794,299191.0625,13.019773,513.804382,2.378974,3.078854,4373797.5,3256546230270.0,-747.128723,131.926346,...,None,None,None,7.664907,None,None,None,None,None,None
21,1745.261963,561793.375,3.427624,7.796992,1.391112,0.173,5098644.5,2572679380990.0,-624.971069,89.92408,...,None,None,None,3.298455,None,None,None,None,None,None


In [32]:
missing_summary = feature_df.isna().sum().sort_values(ascending=False)
missing_summary


mfcc.var[11]                      205
mfcc.var[12]                      205
loudness_ebu128.integrated        205
loudness_ebu128.loudness_range    205
bpm                               205
mfcc.var[1]                       205
mfcc.var[0]                       205
mfcc.var[4]                       205
mfcc.var[5]                       205
mfcc.var[3]                       205
mfcc.var[2]                       205
key_scale                         205
key_key                           205
beats_loudness.mean               205
beats_loudness.var                205
key_strength                      205
mfcc.var[8]                       205
mfcc.var[7]                       205
mfcc.var[6]                       205
mfcc.var[9]                       205
mfcc.var[10]                      205
spectral_centroid.mean              0
spectral_centroid.var               0
spectral_kurtosis.mean              0
spectral_skewness.var               0
spectral_skewness.mean              0
spectral_kur

In [33]:
mfcc_var_cols = [col for col in feature_df.columns if "mfcc.var" in col]
feature_df_clean = feature_df.drop(columns=mfcc_var_cols)


In [34]:
categorical_cols = ["key_key", "key_scale"]

for col in categorical_cols:
    feature_df_clean[col] = feature_df_clean[col].fillna("unknown")


In [35]:
numeric_cols = feature_df_clean.select_dtypes(include=["float64", "int64"]).columns

for col in numeric_cols:
    feature_df_clean[col] = feature_df_clean[col].fillna(feature_df_clean[col].median())


In [38]:
feature_df_clean.dtypes

spectral_centroid.mean            object
spectral_centroid.var             object
spectral_kurtosis.mean            object
spectral_kurtosis.var             object
spectral_skewness.mean            object
spectral_skewness.var             object
spectral_spread.mean              object
spectral_spread.var               object
mfcc.mean[0]                      object
mfcc.mean[1]                      object
mfcc.mean[2]                      object
mfcc.mean[3]                      object
mfcc.mean[4]                      object
mfcc.mean[5]                      object
mfcc.mean[6]                      object
mfcc.mean[7]                      object
mfcc.mean[8]                      object
mfcc.mean[9]                      object
mfcc.mean[10]                     object
mfcc.mean[11]                     object
mfcc.mean[12]                     object
loudness_ebu128.integrated        object
loudness_ebu128.loudness_range    object
dynamic_complexity                object
bpm             

In [39]:
# Alle kolommen behalve de categorische
categorical_cols = ["key_key", "key_scale"]

numeric_cols = [col for col in feature_df_clean.columns if col not in categorical_cols]

# Forceer conversie naar float
for col in numeric_cols:
    feature_df_clean[col] = pd.to_numeric(feature_df_clean[col], errors="coerce")


In [40]:
for col in numeric_cols:
    feature_df_clean[col] = feature_df_clean[col].fillna(feature_df_clean[col].median())


c:\Users\astri\anaconda3\envs\py313\Lib\site-packages\numpy\lib\_nanfunctions_impl.py:1215: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
c:\Users\astri\anaconda3\envs\py313\Lib\site-packages\numpy\lib\_nanfunctions_impl.py:1215: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
c:\Users\astri\anaconda3\envs\py313\Lib\site-packages\numpy\lib\_nanfunctions_impl.py:1215: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
c:\Users\astri\anaconda3\envs\py313\Lib\site-packages\numpy\lib\_nanfunctions_impl.py:1215: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
c:\Users\astri\anaconda3\envs\py313\Lib\site-packages\numpy\lib\_nanfunctions_impl.py:1215: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
c:\Users\astri\anaconda3\envs\py313\Lib\site-packages\numpy\lib\_nanfunctio

In [41]:
feature_df_clean[col].median()


c:\Users\astri\anaconda3\envs\py313\Lib\site-packages\numpy\lib\_nanfunctions_impl.py:1215: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)


np.float64(nan)

In [42]:
empty_cols = feature_df_clean[numeric_cols].isna().all()
empty_cols[empty_cols == True]


loudness_ebu128.integrated        True
loudness_ebu128.loudness_range    True
bpm                               True
beats_loudness.mean               True
beats_loudness.var                True
key_strength                      True
dtype: bool

In [43]:
cols_to_drop = [
    "loudness_ebu128.integrated",
    "loudness_ebu128.loudness_range",
    "bpm",
    "beats_loudness.mean",
    "beats_loudness.var",
    "key_strength"
]

feature_df_clean = feature_df_clean.drop(columns=cols_to_drop)


In [44]:
categorical_cols = ["key_key", "key_scale"]
numeric_cols = [col for col in feature_df_clean.columns if col not in categorical_cols]


In [45]:
# Forceer numeriek
for col in numeric_cols:
    feature_df_clean[col] = pd.to_numeric(feature_df_clean[col], errors="coerce")

# Imputatie
for col in numeric_cols:
    feature_df_clean[col] = feature_df_clean[col].fillna(feature_df_clean[col].median())

# Normalisatie
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
feature_df_clean[numeric_cols] = scaler.fit_transform(feature_df_clean[numeric_cols])


In [46]:
feature_df_clean.head()

,spectral_centroid.mean,spectral_centroid.var,spectral_kurtosis.mean,spectral_kurtosis.var,spectral_skewness.mean,spectral_skewness.var,spectral_spread.mean,spectral_spread.var,mfcc.mean[0],mfcc.mean[1],...,mfcc.mean[6],mfcc.mean[7],mfcc.mean[8],mfcc.mean[9],mfcc.mean[10],mfcc.mean[11],mfcc.mean[12],dynamic_complexity,key_key,key_scale
0,-0.325160,-0.562860,-0.496297,-0.329329,-0.617931,-0.562673,0.957364,-0.330962,0.832235,-0.459936,...,0.506282,0.781787,-0.064080,0.400700,0.000035,-0.014083,0.001894,-0.369181,unknown,unknown
8,-0.271194,-0.180133,-0.402155,-0.319583,-0.497650,-0.502538,-0.051722,-0.319877,0.449429,0.061147,...,0.012564,0.532959,0.216920,-0.425799,-0.095356,0.419944,0.569724,-0.182061,unknown,unknown
14,0.834356,-0.514488,-0.363328,-0.161851,-0.586107,-0.155624,0.602660,-0.337877,1.370433,-0.952494,...,-0.811526,0.086285,-0.143250,-0.193565,0.787981,-0.192305,-0.556768,-0.994769,unknown,unknown
17,-0.802479,-0.565870,-0.121310,-0.123657,-0.079803,0.245904,0.163690,-0.439734,-0.748835,-0.427187,...,0.720497,-0.509857,-0.633402,-0.738639,-0.022800,-0.445941,0.295811,1.645470,unknown,unknown
21,1.763650,0.018912,-0.558285,-0.334054,-0.898817,-0.591077,0.875495,-0.543274,1.228529,-1.820719,...,-0.879332,0.868219,-0.333562,-0.575944,0.295389,0.068193,0.254984,-0.637572,unknown,unknown


In [48]:
# Metadata kolommen die we willen behouden
metadata_cols = ["track_name", "artist_name", "playlist_name", "owner", "mbid"]

# Metadata subset
metadata_df = work_df_with_ab[metadata_cols]

# Samenvoegen op index
final_df = metadata_df.join(feature_df_clean)

final_df.head()


,track_name,artist_name,playlist_name,owner,mbid,spectral_centroid.mean,spectral_centroid.var,spectral_kurtosis.mean,spectral_kurtosis.var,spectral_skewness.mean,...,mfcc.mean[6],mfcc.mean[7],mfcc.mean[8],mfcc.mean[9],mfcc.mean[10],mfcc.mean[11],mfcc.mean[12],dynamic_complexity,key_key,key_scale
0,'74-'75,The Connells,NaN,Astrid,bcb4d5d0-e8e2-4284-8a62-ec74f808a5de,-0.325160,-0.562860,-0.496297,-0.329329,-0.617931,...,0.506282,0.781787,-0.064080,0.400700,0.000035,-0.014083,0.001894,-0.369181,unknown,unknown
8,A Song for the Drunk and Broken Hearted,Passenger,NaN,Astrid,434dbfe7-a437-4030-87aa-36f020099909,-0.271194,-0.180133,-0.402155,-0.319583,-0.497650,...,0.012564,0.532959,0.216920,-0.425799,-0.095356,0.419944,0.569724,-0.182061,unknown,unknown
14,Absolute,The Fray,Dear Diary,Astrid,04b1bd97-5039-439f-ab4d-5e50d8b0a51d,0.834356,-0.514488,-0.363328,-0.161851,-0.586107,...,-0.811526,0.086285,-0.143250,-0.193565,0.787981,-0.192305,-0.556768,-0.994769,unknown,unknown
17,After Rain,Dermot Kennedy,NaN,Astrid,ac0abc3e-5aff-4baf-ad60-a1a66fad27cb,-0.802479,-0.565870,-0.121310,-0.123657,-0.079803,...,0.720497,-0.509857,-0.633402,-0.738639,-0.022800,-0.445941,0.295811,1.645470,unknown,unknown
21,Ain't No Rest for the Wicked,Cage The Elephant,NaN,Astrid,e9bb1073-94e7-4ad0-a623-0bf0034af48c,1.763650,0.018912,-0.558285,-0.334054,-0.898817,...,-0.879332,0.868219,-0.333562,-0.575944,0.295389,0.068193,0.254984,-0.637572,unknown,unknown
